# Feature Engineering & Data Analysis

**Continuing from:** `exploring_cleaning_data.ipynb` — this notebook picks up once the data is *clean*, and turns it into something that can actually answer business questions.

## Why this phase exists

Cleaning makes data **correct** — no missing values, no duplicates, the right dtypes. That's necessary, but it's not the same as data being **useful**. A clean `Order Date` column doesn't tell you whether Mondays sell more than Fridays; a clean `Sales`/`Profit` pair doesn't tell you which customers are actually worth the most. Feature engineering is the step where we *derive* new columns that carry that signal explicitly, and data analysis is the step where we use those columns (plus the originals) to actually answer questions.

Skipping straight from cleaning to charts is a common shortcut — and it produces charts that answer questions nobody asked, because the underlying features were never designed around a question in the first place. This notebook keeps the two steps in order: **know the question → build the feature that answers it → analyze it.**

## The Generic Workflow

This is the same repeatable process regardless of the dataset — the checklist we'll apply to the Superstore data in Part 2 below.

0. **Setup & Recap** — load the cleaned data, confirm it survived the handoff intact.
1. **Define the Analysis Questions** — write down what we're trying to answer *before* building features, so every feature has a reason to exist.
2. **Feature Engineering — Derived Columns** — transform existing columns (dates, numeric ratios, categorical cleanup) into new per-row signal.
3. **Feature Engineering — Aggregated Features** — roll row-level data up into entity-level features (per customer, per product, etc.).
4. **Feature Validation** — sanity-check every new feature immediately: nulls introduced by the transform, implausible ranges, divide-by-zero.
5. **Exploratory Data Analysis** — univariate → bivariate → multivariate, in that order.
6. **Answering the Questions** — go back to Step 1 and directly resolve each question with a specific table or aggregation.
7. **Insight Synthesis** — summarize findings in plain language, including caveats inherited from cleaning decisions.
8. **Handoff Prep** — name and preserve the tables/features the next stage (visualization) will need.

## Part 2 — Applying the Workflow to the Superstore Dataset

From here on, every section is Part 2: the same nine steps, made concrete for `Sample-Superstore2019.csv`.

> **Note on notebooks and kernels:** this notebook does **not** share memory with `exploring_cleaning_data.ipynb` — each `.ipynb` file runs its own kernel. Step 0 below re-loads the raw CSV and re-applies the cleaning decisions already validated there, rather than assuming `df` already exists. This keeps the notebook runnable on its own.

### Step 0 — Setup & Recap

We re-establish the clean baseline before building anything new. Nothing new is decided here, we're just reproducing what the cleaning notebook already validated:

- Drop `Unnamed: 0` and `Row ID` (load artifacts, not data)
- Convert `Order Date` / `Ship Date` to real `datetime`
- Fill the 11 missing `Postal Code` values (all Burlington, VT) with `05401`
- Convert low-cardinality text columns to `category`
- Drop the single true exact-duplicate row

One deliberate omission: the 7 `Order ID` + `Product ID` pairs that share a combination but have different `Quantity`/`Sales` are **not** collapsed. Investigation in the cleaning notebook showed each pair has matching unit price and unit profit — they're legitimate separate order lines, not data-entry errors, and dropping either row would silently discard real revenue.

In [2]:
# Importing the proper and required data packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# Reading the clean data and make a copy of the original file to be the single source of truth
df = pd.read_csv("Sample_Superstore_2019_Clean.csv")
df.shape

(9993, 20)

In [ ]:
# Creating the copy of the original DataFrame
df_clean = df.copy()

### Step 1 — Define the Analysis Questions

These are the questions the rest of this notebook is built to answer. Every feature engineered in Steps 2–3 exists to serve at least one of these:

1. Which `State` generates the most total `Profit`, and which generates the biggest total loss?
2. Is there a relationship between `Discount` and `Profit`?
3. Does `Ship Mode` relate to profitability or shipping time?
4. Is there seasonality in `Sales` — by month, or by day of week?
5. Who are the most valuable customers, and how would we segment them?

Step 6 will come back to this exact list and answer each one directly.

### Step 2 — Feature Engineering: Derived Columns

Two families of per-row features, built directly from existing columns:

- **Date-based**, from `Order Date` / `Ship Date`: how long shipping took, and what calendar pattern the order falls into (weekday, month, quarter, year, weekend flag) — needed to test the seasonality question (Q4).
- **Numeric transforms**, from `Sales` / `Quantity` / `Discount`: a per-unit price, a profit-margin ratio, and a discount bucket — needed to test the discount question (Q2).

In [ ]:
# Date-based features -- needed for the seasonality question (Q4)
df["Shipping Days"] = (df["Ship Date"] - df["Order Date"]).dt.days
df["Order Weekday"] = df["Order Date"].dt.day_name()
df["Order Month"] = df["Order Date"].dt.month
df["Order Quarter"] = df["Order Date"].dt.quarter
df["Order Year"] = df["Order Date"].dt.year
df["Is Weekend"] = df["Order Date"].dt.dayofweek >= 5

df[["Order Date", "Ship Date", "Shipping Days", "Order Weekday", "Is Weekend"]].head()

In [ ]:
# Numeric transforms -- needed for the discount question (Q2)
df["Unit Price"] = df["Sales"] / df["Quantity"]
df["Profit Margin"] = df["Profit"] / df["Sales"]

# Discount buckets: 0 = no discount, then Low/Medium/High bands.
# Bins chosen from the actual observed range (0.0 to 0.8).
df["Discount Bucket"] = pd.cut(
    df["Discount"],
    bins=[-0.01, 0, 0.2, 0.4, 1.0],
    labels=["None", "Low", "Medium", "High"],
)

df[["Sales", "Quantity", "Unit Price", "Profit", "Profit Margin", "Discount", "Discount Bucket"]].head()

### Step 3 — Feature Engineering: Aggregated Features

Row-level features answer row-level questions. To answer "who are our most valuable customers?" (Q5) we need to roll individual order lines up to one row per entity:

- **Customer-level**: how often they order, how much they spend, how long they've been a customer
- **Product-level**: how much of each product sells, and at what average discount
- **RFM** (Recency / Frequency / Monetary): the classic customer-value framework, built directly from the two aggregates above plus `Order Date`

In [ ]:
# One row per customer: order count, spend, average order value, tenure
customer_features = df.groupby("Customer ID", observed=True).agg(
    order_count=("Order ID", "nunique"),
    total_spend=("Sales", "sum"),
    total_profit=("Profit", "sum"),
    avg_order_value=("Sales", "mean"),
    first_order=("Order Date", "min"),
    last_order=("Order Date", "max"),
)

# Tenure = span between a customer's first and last order in this dataset
customer_features["Customer Tenure Days"] = (
    customer_features["last_order"] - customer_features["first_order"]
).dt.days

customer_features.sort_values("total_spend", ascending=False).head()

In [ ]:
# One row per product: units sold, revenue, average discount given
product_features = df.groupby("Product ID", observed=True).agg(
    total_qty_sold=("Quantity", "sum"),
    total_revenue=("Sales", "sum"),
    avg_discount=("Discount", "mean"),
).sort_values("total_revenue", ascending=False)

product_features.head()

In [ ]:
# RFM: Recency (days since last order), Frequency (distinct orders),
# Monetary (total Sales). Recency is measured against the day after the
# last order in the whole dataset, since we have no "today" to measure from.
snapshot_date = df["Order Date"].max() + pd.Timedelta(days=1)

rfm = df.groupby("Customer ID", observed=True).agg(
    Recency=("Order Date", lambda x: (snapshot_date - x.max()).days),
    Frequency=("Order ID", "nunique"),
    Monetary=("Sales", "sum"),
)

rfm.sort_values("Monetary", ascending=False).head()

### Step 4 — Feature Validation

New features can silently break things the raw columns never would: a ratio can divide by zero, a date subtraction can go negative if the source dates were wrong, a bin can leave rows unlabeled. Check every new feature once, right after creating it.

In [ ]:
# Divide-by-zero / infinite values in the ratio features
print("Infinite Unit Price values:", np.isinf(df["Unit Price"]).sum())
print("Infinite Profit Margin values:", np.isinf(df["Profit Margin"]).sum())

# Shipping Days should never be negative -- that would mean a product
# shipped before it was ordered
print("Shipping Days range:", df["Shipping Days"].min(), "to", df["Shipping Days"].max())
print("Negative Shipping Days:", (df["Shipping Days"] < 0).sum())

# Every row should have landed in exactly one Discount Bucket
print("Unlabeled Discount Bucket rows:", df["Discount Bucket"].isnull().sum())

### Step 5 — Exploratory Data Analysis

Univariate first (what does one feature look like on its own), then bivariate (how does it relate to profit), then multivariate (multiple dimensions at once).

In [ ]:
# Univariate: distribution of the new per-row features
print(df["Discount Bucket"].value_counts())
print()
print(df["Shipping Days"].describe())
print()
print(df["Profit Margin"].describe())

In [ ]:
# Bivariate: Profit against Discount Bucket, Sales against calendar features
print(df.groupby("Discount Bucket", observed=True)["Profit"].mean())
print()
print(df.groupby("Order Month", observed=True)["Sales"].sum())

**Multivariate**: once two dimensions aren't enough, `.groupby()` on multiple columns and `.pivot_table()` spread one categorical column across the columns of the result — closer to the shape a report or chart actually needs.

- `.groupby("column")` splits the DataFrame into groups sharing the same value; chaining an aggregation (`.sum()`, `.mean()`, `.agg()`) collapses each group into one row.
- `.pivot_table()` is the more flexible version — it lets a second categorical column spread across the result's columns.
- `.sort_values()` orders the result so the most/least interesting rows are easy to spot.

In [ ]:
# Total profit and sales per Region, sorted from most to least profitable
region_summary = df.groupby("Region", observed=True)[["Sales", "Profit"]].sum().sort_values("Profit", ascending=False)
region_summary

In [ ]:
# Multiple aggregations at once with .agg()
category_summary = df.groupby("Category", observed=True).agg(
    total_sales=("Sales", "sum"),
    avg_profit=("Profit", "mean"),
    orders=("Order ID", "count"),
)
category_summary

In [ ]:
# pivot_table: average Sales by Region (rows) x Category (columns)
pivot = pd.pivot_table(
    df,
    values="Sales",
    index="Region",
    columns="Category",
    aggfunc="mean",
    observed=True,
)
pivot

### Step 6 — Answering the Questions

Back to the five questions from Step 1 — each one resolved directly with the features built above.

In [ ]:
# Q1: which State generates the most Profit, and which the biggest loss?
state_profit = df.groupby("State", observed=True)["Profit"].sum().sort_values(ascending=False)
print("Highest-profit state:", state_profit.idxmax(), "->", round(state_profit.max(), 2))
print("Biggest-loss state:", state_profit.idxmin(), "->", round(state_profit.min(), 2))

In [ ]:
# Q2: relationship between Discount and Profit, using the bucket built in Step 2
discount_profit = df.groupby("Discount Bucket", observed=True)["Profit"].mean()
discount_profit

In [ ]:
# Q3: does Ship Mode relate to profitability or shipping time?
ship_mode_summary = df.groupby("Ship Mode", observed=True).agg(
    avg_profit=("Profit", "mean"),
    avg_shipping_days=("Shipping Days", "mean"),
)
ship_mode_summary

In [ ]:
# Q4: seasonality in Sales -- by month and by weekday
print(df.groupby("Order Month", observed=True)["Sales"].sum())
print()
print(df.groupby("Order Weekday", observed=True)["Sales"].sum().sort_values(ascending=False))

In [ ]:
# Q5: most valuable customers, using the RFM features from Step 3
rfm.sort_values("Monetary", ascending=False).head()

### Try It Yourself

1. Group by `Sub-Category` and find total `Profit`, sorted ascending (most loss-making first) — which sub-category loses the most money overall?
2. Build a pivot table showing total `Sales` with `Segment` as rows and `Ship Mode` as columns.
3. Using `.groupby()` with `.agg()`, compute both the average and the maximum `Shipping Days` per `Ship Mode`.

In [ ]:
# TODO 1: total Profit per Sub-Category, sorted ascending


# TODO 2: pivot table -- Sales by Segment (rows) x Ship Mode (columns)


# TODO 3: average and max Shipping Days per Ship Mode

### Step 7 — Insight Synthesis

- **Profit by state**: California is the strongest performer (~$76.4K total profit); Texas is the biggest drag (~-$25.7K total loss) — worth checking whether that's driven by discounting practices specific to Texas.
- **Discount vs. Profit**: a clear, monotonic relationship. Average profit per order falls as the discount band rises — roughly +$67 with no discount, +$27 at Low, **-$78 at Medium, -$107 at High**. Discounts above ~20% are, on average, selling at a loss.
- **Ship Mode**: shipping time behaves exactly as expected (Same Day ≈ 0 days, Standard Class ≈ 5 days), but average profit per order is fairly flat across all four modes (~$28–32) — shipping choice doesn't appear to drive profitability on its own.
- **Seasonality**: Sales rise sharply toward year-end (November and December are the two highest months, roughly 3–4x February's total), consistent with holiday retail patterns. By weekday, Monday/Tuesday/Wednesday outsell Friday by close to 2x.
- **Customer value**: spend is concentrated — the top RFM customer alone accounts for ~$25K in lifetime Sales, well above the ~$2.9K average, suggesting a small set of high-value accounts worth treating differently from the broader base.

**Caveat carried from cleaning:** these Profit/Sales totals reflect the decision *not* to collapse the 7 legitimate `Order ID` + `Product ID` duplicate pairs — collapsing them would have understated every total above.

### Step 8 — Handoff Prep for Visualization

The next stage (lectures 7–8) turns these tables into charts. This notebook's job is to make sure nothing needs recomputing there — everything it needs already exists, by name, above.

In [ ]:
# Tables and features the visualization stage will need -- named here so
# nothing has to be recomputed in the next notebook.
handoff_manifest = {
    "df": "row-level cleaned + feature-engineered DataFrame",
    "customer_features": "one row per Customer ID -- spend, order count, tenure",
    "product_features": "one row per Product ID -- units sold, revenue, avg discount",
    "rfm": "one row per Customer ID -- Recency, Frequency, Monetary",
    "state_profit": "total Profit per State",
    "discount_profit": "average Profit per Discount Bucket",
    "ship_mode_summary": "average Profit and Shipping Days per Ship Mode",
    "region_summary": "total Sales and Profit per Region",
    "category_summary": "total Sales, avg Profit, order count per Category",
    "pivot": "average Sales by Region x Category",
}
for name, description in handoff_manifest.items():
    print(f"{name}: {description}")